In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [4]:
CSV_PATH = 'C:/Users/mohad/OneDrive/Desktop/dataset/raw_car_dataset.csv'
df = pd.read_csv(CSV_PATH)

In [5]:
print("\n=== STEP 1: LOAD & INSPECT ===")
print("\nHead (10 rows):")
print(df.head(10))


=== STEP 1: LOAD & INSPECT ===

Head (10 rows):
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [12]:
print("Shape:")
print(df.shape)

Shape:
(145, 6)


In [14]:
print("===Info:===")
print(df.info())

===Info:===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


In [16]:
print("Missing Values:\n")
print(df.isnull().sum())

Missing Values:

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [17]:
print("\nLocation Value Counts:")
print(df["Location"].value_counts(dropna=False))


Location Value Counts:
Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [18]:
print("\nIssues Found: Missing values, typos in Location, mixed Price formats, duplicates, outliers")


Issues Found: Missing values, typos in Location, mixed Price formats, duplicates, outliers


In [19]:
print("\n=== STEP 2: CLEAN PRICE ===")


=== STEP 2: CLEAN PRICE ===


In [20]:
df["Price"] = df["Price"].replace(r"[$,]", "", regex=True).astype(float)

In [21]:
print("Price skewness:", df["Price"].skew())

Price skewness: 7.871113660850301


In [22]:
print("\n=== STEP 3: FIX LOCATION ERRORS ===")


=== STEP 3: FIX LOCATION ERRORS ===


In [23]:
df["Location"] = df["Location"].str.strip().str.title()

In [24]:
df["Location"] = df["Location"].replace({
"Subrb": "Suburb",
"Cty": "City",
"??": np.nan,
"Unknown": np.nan
})

In [25]:
print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    53
Rural     21
NaN       12
Name: count, dtype: int64


In [26]:
print("\n=== STEP 4: IMPUTATION ===")


=== STEP 4: IMPUTATION ===


In [27]:
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

In [31]:
print("Missing after imputation:")
print(df.isnull().sum())

Missing after imputation:
Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [32]:
print("\n=== STEP 5: REMOVE DUPLICATES ===")

before = df.shape
df = df.drop_duplicates()
after = df.shape

print("Before:", before)
print("After:", after)
print("Rows removed:", before[0] - after[0])


=== STEP 5: REMOVE DUPLICATES ===
Before: (145, 6)
After: (140, 6)
Rows removed: 5


In [38]:
print("\n=== STEP 6: OUTLIERS ===")

def iqr_bounds(series):
    
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

p_low, p_high = iqr_bounds(df["Price"])
o_low, o_high = iqr_bounds(df["Odometer_km"])

df["Price"] = df["Price"].clip(p_low, p_high)
df["Odometer_km"] = df["Odometer_km"].clip(o_low, o_high)

print(df[["Price", "Odometer_km"]].describe())


=== STEP 6: OUTLIERS ===
             Price    Odometer_km
count   140.000000     140.000000
mean   3175.456250  130945.403571
std    2601.848629   53815.006935
min    1500.000000    5000.000000
25%    1500.000000   97844.000000
50%    1500.000000  128548.000000
75%    4489.750000  167501.500000
max    8974.375000  271987.750000


In [39]:
print("\n=== STEP 7: ENCODING ===")

df = pd.get_dummies(df, columns=["Location"], dtype=int)

print("New columns:")
print([col for col in df.columns if "Location_" in col])


=== STEP 7: ENCODING ===
New columns:
['Location_City', 'Location_Rural', 'Location_Suburb']


In [40]:
print("\n=== STEP 8: FEATURE ENGINEERING ===")

CURRENT_YEAR = 2025

df["CarAge"] = CURRENT_YEAR - df["Year"]

df["Km_per_year"] = df["Odometer_km"] / df["CarAge"].replace(0, np.nan)

df["Is_Urban"] = df.get("Location_City", 0)

df["LogPrice"] = np.log1p(df["Price"])

print(df[["CarAge", "Km_per_year", "Is_Urban", "LogPrice"]].head())


=== STEP 8: FEATURE ENGINEERING ===
   CarAge   Km_per_year  Is_Urban  LogPrice
0      27   5104.814815         1  7.313887
1       9  14283.111111         0  8.336151
2      11   9754.727273         0  8.581482
3      26   5455.307692         0  7.313887
4       3  42849.333333         1  7.313887


In [41]:
print("\n=== STEP 9: SCALING ===")

dont_scale = ["Price", "LogPrice"]

dummy_cols = [col for col in df.columns if col.startswith("Location_")]

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

scale_cols = [col for col in numeric_cols if col not in dont_scale and col not in dummy_cols]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(df[scale_cols].head())


=== STEP 9: SCALING ===
   Odometer_km     Doors  Accidents      Year    CarAge  Km_per_year  Is_Urban
0     0.128390  0.254091   0.316968 -1.686714  1.686714    -0.553364  1.028992
1    -0.044709  0.254091  -0.820867  0.794617 -0.794617    -0.003812 -0.971825
2    -0.440923  0.254091  -0.820867  0.518913 -0.518913    -0.274949 -0.971825
3     0.203135  0.254091   0.316968 -1.548862  1.548862    -0.532378 -0.971825
4    -0.044709 -0.931668  -0.820867  1.621727 -1.621727     1.706594  1.028992


In [42]:
print("\n=== STEP 10: FINAL CHECK ===")

print("\nFinal Info:")
print(df.info())

print("\nMissing Values (should be 0):")
print(df.isnull().sum())

print("\nDescribe:")
print(df.describe())


=== STEP 10: FINAL CHECK ===

Final Info:
<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Suburb  140 non-null    int64  
 8   CarAge           140 non-null    float64
 9   Km_per_year      140 non-null    float64
 10  Is_Urban         140 non-null    float64
 11  LogPrice         140 non-null    float64
dtypes: float64(9), int64(3)
memory usage: 14.2 KB
None

Missing Values (should be 0):
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0


In [43]:
assert df["Price"].dtype == float
assert "LogPrice" in df.columns
assert df.isnull().sum().sum() == 0
assert any(col.startswith("Location_") for col in df.columns)

print("\nAll sanity checks passed!")


All sanity checks passed!


In [44]:
df.to_csv("clean_car_dataset.csv", index=False)

print("\nFile saved as clean_car_dataset.csv")


File saved as clean_car_dataset.csv
